**PS2 — Cascade Chain Pattern Analysis**

---
SageMaker Studio | MLflow | Feature Store

Architecture : CUBIC MARS AWS v6
Gold table   : mars_dev.gold.device_ps2_chains
Grain        : one row per (device_id, transit_day) where chain_length >= 2
               — cascade days only; non-cascade days are NOT in this table

Purpose      : Understand HOW cascades propagate — not predict them.
               Time windows (0-5 / 5-15 / 15-30 / 30-60 / 60+ min)
               reveal propagation speed, subsystem co-failure patterns,
               and severity regime transitions per device category.

Analysis     : Time-window propagation | Phi correlation | Conditional P(B|A,T)
               Markov chain | HMM 3-state | Association rules | Network graph
Device scope : TVM | GATE (RVG/HBG/SAG) | VALIDATOR (BMV bus)
Feasibility  : 85%  (15% gap: SVN_STAGE.CMDB_CI / CMDB_REL = 0 rows)

---

In [ ]:
# ── CELL 1 : Install dependencies ─────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs", "pyarrow",
    "scikit-learn>=1.4", "xgboost>=2.0", "lightgbm>=4.3",
    "hmmlearn>=0.3", "mlxtend>=0.23",
    "optuna>=3.6", "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0",
    "pandas>=2.0", "matplotlib", "seaborn", "networkx", "scipy",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
import boto3, sagemaker, joblib
import scipy.stats as stats

from collections import Counter, defaultdict
from itertools import combinations

from hmmlearn  import hmm
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing     import TransactionEncoder

import mlflow
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
# Analysis-only notebook — no SageMaker endpoint or model registry.
# Outputs: MLflow experiment with artifact charts, CSVs, matrices.

ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET     = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX     = "chicago/gold/device_ps2_chains"
GOLD_S3         = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION = Session()
ROLE       = get_execution_role()
REGION     = boto3.Session().region_name
SM_CLIENT  = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT  = boto3.client("s3",        region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)

EXPERIMENT_NAME = "chicago-ps2-cascade-analysis"
FS_GROUP_NAME   = "chicago-ps2-features"

# Time-window buckets for cascade propagation analysis
TIME_WINDOWS = {
    "0-5 min"  : (0,   5),
    "5-15 min" : (5,  15),
    "15-30 min": (15, 30),
    "30-60 min": (30, 60),
    "60+ min"  : (60, 9999),
}

DEVICE_CATEGORIES = ["TVM", "GATE", "VALIDATOR"]

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3 : {GOLD_S3}")
print(f"Time windows: {list(TIME_WINDOWS.keys())}")

In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
print(f"Loading Gold table from: {GOLD_S3}")

df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

df["transit_day"] = pd.to_datetime(df["transit_day"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['transit_day'].min().date()}  →  {df['transit_day'].max().date()}")
print(f"\nCascade type distribution (% of all cascade days):")
for col in ["has_cash_cascade","has_gate_mech_in_chain","has_printer_in_chain",
            "has_csc_reader_in_chain","has_dopp_in_chain","has_scrst_in_chain"]:
    if col in df.columns:
        print(f"  {col:<30}: {df[col].mean():.2%}")
print(f"\nChain length stats:\n{df['chain_length'].describe().round(2)}")
print(f"\nData source: {DATA_SOURCE}")

In [ ]:
# ── CELL 5 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Cascade type prevalence
type_cols = [c for c in ["has_cash_cascade","has_gate_mech_in_chain",
                          "has_printer_in_chain","has_csc_reader_in_chain",
                          "has_dopp_in_chain","has_scrst_in_chain"]
             if c in df.columns]
if type_cols:
    df[type_cols].mean().sort_values(ascending=False).plot(
        kind="bar", ax=axes[0,0], color="#1565C0", edgecolor="white")
    axes[0,0].set_title("Cascade type prevalence")
    axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=45, ha="right")
    axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y,_: f"{y:.0%}"))

# Chain length distribution
df["chain_length"].plot(kind="hist", ax=axes[0,1], bins=20,
                        color="#43A047", edgecolor="white")
axes[0,1].set_title("Chain length distribution")
axes[0,1].set_xlabel("chain_length (faults per cascade day)")

# Cascade days over time
df.groupby("transit_day").size().rolling(7).mean().plot(
    ax=axes[0,2], color="#E53935", linewidth=0.8)
axes[0,2].set_title("Cascade days (7-day rolling avg)")

# Chain span distribution (minutes)
if "chain_span_min" in df.columns:
    df["chain_span_min"].clip(upper=120).plot(
        kind="hist", ax=axes[1,0], bins=30, color="#7B1FA2", edgecolor="white")
    axes[1,0].set_title("Chain span distribution (min, clipped @120)")
    axes[1,0].set_xlabel("chain_span_min")

# Device category breakdown
if "mars_device_category" in df.columns:
    df["mars_device_category"].value_counts().plot(
        kind="bar", ax=axes[1,1], color="#E65100", edgecolor="white")
    axes[1,1].set_title("Cascade days by device category")
    axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=0)

# Chain length by device category
if "mars_device_category" in df.columns:
    df.boxplot(column="chain_length", by="mars_device_category",
               ax=axes[1,2], patch_artist=True)
    axes[1,2].set_title("Chain length by device category")
    axes[1,2].set_xlabel("")
    plt.sca(axes[1,2]); plt.title("Chain length by device category")

plt.suptitle("PS2 — Cascade Chain EDA", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("/tmp/ps2_eda.png", dpi=100); plt.show()
print("EDA complete.")

In [ ]:
# ── CELL 6 : Feature engineering & temporal splits ───────────────────────────
# Analysis grain: (device_id, transit_day) cascade days only (chain_length >= 2).
# No prediction target — this is a pure analysis notebook.
# We derive a time-window bucket column from chain_span_min for downstream analysis.

SECONDARY_TARGETS = ["has_cash_cascade","has_gate_mech_in_chain",
                     "has_printer_in_chain","has_csc_reader_in_chain"]

EXCLUDE_COLS = [
    "DEVICE_ID","DEVICE_KEY","transit_day","mars_device_category",
    "DEVICE_NAME","FACILITY_ID","FACILITY_NAME","OPERATOR_ID","OPERATOR_NAME",
    "DEVICE_SERIAL_NUMBER","DEVICE_CONTROL_GROUP_TYPE_NAME","BUS_ID",
    "subsystem_chain","event_type_chain","severity_chain",
    "chain_start_dtm","chain_end_dtm",
    "first_subsystem","last_subsystem",
    "ci_related_devices","shared_facility_chain","svn_ci_data_available",
    "has_cascade",
] + SECONDARY_TARGETS

FEATURE_COLS = [
    c for c in df.columns
    if c not in EXCLUDE_COLS
    and pd.api.types.is_numeric_dtype(df[c])
]
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())

# Temporal split (kept for Feature Store + HMM sequencing)
train_df = df[df["transit_day"] <  "2025-10-01"]
val_df   = df[(df["transit_day"] >= "2025-10-01") & (df["transit_day"] < "2026-01-01")]
test_df  = df[df["transit_day"] >= "2026-01-01"]

# Time-window bucket derived from chain_span_min
def assign_window(span):
    for label, (lo, hi) in TIME_WINDOWS.items():
        if lo <= span < hi:
            return label
    return "60+ min"

if "chain_span_min" in df.columns:
    df["window_bucket"] = df["chain_span_min"].apply(assign_window)
    window_order = list(TIME_WINDOWS.keys())
    df["window_bucket"] = pd.Categorical(df["window_bucket"],
                                          categories=window_order, ordered=True)

# Cascade velocity: average minutes per additional fault in chain
if "chain_span_min" in df.columns:
    df["cascade_velocity_min_per_fault"] = (
        df["chain_span_min"] / (df["chain_length"] - 1).clip(lower=1))

print(f"Features        : {len(FEATURE_COLS)}")
print(f"Train rows      : {len(train_df):,}  ({train_df['transit_day'].min().date()} → {train_df['transit_day'].max().date()})")
print(f"Val rows        : {len(val_df):,}  ({val_df['transit_day'].min().date()} → {val_df['transit_day'].max().date()})")
print(f"Test rows       : {len(test_df):,}  ({test_df['transit_day'].min().date()} → {test_df['transit_day'].max().date()})")

if "window_bucket" in df.columns:
    print(f"\n── Cascade time-window distribution ───────────────────────────────────")
    wc = df["window_bucket"].value_counts().sort_index()
    for w, n in wc.items():
        print(f"  {w:<12} : {n:>6,} cascade days  ({n/len(df):.1%})")

In [ ]:
# ── CELL 7 : Feature Store ────────────────────────────────────────────────────
# Ingests cascade chain features into SageMaker Feature Store (offline + online).
# IAM note: requires AmazonSageMakerFeatureStoreAccess on cubic-mars-role-sagemaker-exec-dev.
# This cell will fail gracefully if that policy is not yet attached.

fs_cols = FEATURE_COLS + SECONDARY_TARGETS + ["DEVICE_ID", "transit_day"]
fs_cols = [c for c in fs_cols if c in train_df.columns]
fs_df = train_df[fs_cols].copy()
fs_df["event_time"] = fs_df["transit_day"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
fs_df               = fs_df.drop(columns=["transit_day"])

def dtype_to_fs(dtype):
    if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
    if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
    return FeatureTypeEnum.STRING

feature_group = FeatureGroup(
    name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
    feature_definitions=[
        FeatureDefinition(feature_name=c, feature_type=dtype_to_fs(fs_df[c].dtype))
        for c in fs_df.columns
    ])
try:
    feature_group.create(
        s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
        record_identifier_name="DEVICE_ID",
        event_time_feature_name="event_time",
        role_arn=ROLE, enable_online_store=True,
        description="PS2 cascade chain features — Chicago Ventra")
    status = ""
    while status != "Created":
        time.sleep(15)
        status = feature_group.describe().get("FeatureGroupStatus")
        print(f"  {status}")
    print(f"Feature Store group created: {FS_GROUP_NAME}")
except Exception as e:
    print(f"Feature Group (skipping if not yet authorized): {e}")

In [ ]:
# ── CELL 8 : Ingest to Feature Store ─────────────────────────────────────────
feature_group.ingest(data_frame=fs_df.reset_index(drop=True), max_workers=4, wait=True)
print(f"Ingested {len(fs_df):,} records → {FS_GROUP_NAME}")

In [ ]:
# ── CELL 9 : Helper functions ─────────────────────────────────────────────────
COMMON_TAGS = {
    "ps"           : "PS2",
    "city"         : "chicago",
    "gold_version" : str(GOLD_VERSION),
    "architecture" : "CUBIC-MARS-AWS-v6",
    "analysis_type": "cascade_pattern_analysis",
}

def parse_chain(chain_str, sep="->"):
    """Parse 'BHU->CHU->PRINTER' into ['BHU','CHU','PRINTER']."""
    if pd.isna(chain_str) or not chain_str:
        return []
    return [s.strip() for s in str(chain_str).split(sep) if s.strip()]

def get_all_subsystems(df_sub, chain_col="subsystem_chain"):
    """Return sorted list of all unique subsystems across all chains."""
    all_subs = set()
    for chain in df_sub[chain_col].dropna():
        all_subs.update(parse_chain(chain))
    return sorted(all_subs)

def chain_to_binary_flags(df_sub, subsystems, chain_col="subsystem_chain"):
    """Return binary DataFrame: rows=cascade days, cols=subsystems (1 if present)."""
    rows = []
    for chain in df_sub[chain_col]:
        present = set(parse_chain(chain))
        rows.append({s: int(s in present) for s in subsystems})
    return pd.DataFrame(rows, index=df_sub.index)

def phi_corr(x, y):
    """Phi (Matthews) correlation coefficient for two binary series."""
    n   = len(x)
    n11 = (x & y).sum(); n10 = (x & ~y).sum()
    n01 = (~x & y).sum(); n00 = (~x & ~y).sum()
    n1_ = n11 + n10; n0_ = n01 + n00
    n_1 = n11 + n01; n_0 = n10 + n00
    denom = np.sqrt(n1_ * n0_ * n_1 * n_0)
    return (n * n11 - n1_ * n_1) / denom if denom > 0 else 0.0

def save_and_log(fig, path, run=None):
    """Save figure and optionally log to active MLflow run."""
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.show(); plt.close(fig)
    if run is not None:
        mlflow.log_artifact(path)
    return path

In [ ]:
# ── CELL 10 : Cascade Propagation — Time-Window Analysis ─────────────────────
# Buckets: 0-5 / 5-15 / 15-30 / 30-60 / 60+ minutes (chain_span_min)
# For each window: chain_length, cascade velocity, subsystem mix, device category.
# Answers: how fast does a cascade propagate and what does each speed regime look like?

if "window_bucket" not in df.columns:
    print("chain_span_min not available — time-window analysis skipped")
else:
    with mlflow.start_run(run_name="cascade_time_window_analysis") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "time_window_analysis"})
        mlflow.log_param("time_windows", str(list(TIME_WINDOWS.keys())))

        window_order = list(TIME_WINDOWS.keys())

        # ── 1. Chain length by window ─────────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        wg = df.groupby("window_bucket", observed=True)["chain_length"]
        wg.mean().reindex(window_order).plot(kind="bar", ax=axes[0],
            color="#1565C0", edgecolor="white")
        axes[0].set_title("Mean chain length per time window")
        axes[0].set_xlabel(""); axes[0].set_ylabel("Avg faults in cascade")
        axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha="right")

        df.boxplot(column="chain_length", by="window_bucket",
                   ax=axes[1], patch_artist=True,
                   positions=range(len(window_order)),
                   labels=window_order)
        axes[1].set_title("Chain length distribution by time window")
        axes[1].set_xlabel(""); plt.sca(axes[1]); plt.title("Chain length by window")
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha="right")

        plt.suptitle("Cascade Propagation — Fault Count vs Speed", fontsize=11)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_window_chain_length.png", run)

        # ── 2. Cascade velocity by window and device category ─────────────────
        if "cascade_velocity_min_per_fault" in df.columns and "mars_device_category" in df.columns:
            fig, ax = plt.subplots(figsize=(12, 5))
            vel_pivot = (df.groupby(["window_bucket","mars_device_category"],
                                     observed=True)["cascade_velocity_min_per_fault"]
                           .mean()
                           .unstack("mars_device_category")
                           .reindex(window_order))
            vel_pivot.plot(kind="bar", ax=ax, edgecolor="white")
            ax.set_title("Cascade velocity (min/fault) by time window and device category")
            ax.set_xlabel("Time window")
            ax.set_ylabel("Minutes per additional fault")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
            ax.legend(title="Device category")
            plt.tight_layout()
            save_and_log(fig, "/tmp/ps2_cascade_velocity.png", run)

        # ── 3. Window distribution by device category (stacked bar) ───────────
        if "mars_device_category" in df.columns:
            fig, ax = plt.subplots(figsize=(10, 5))
            cat_window = (df.groupby(["mars_device_category","window_bucket"],
                                      observed=True)
                            .size().unstack("window_bucket", fill_value=0)
                            .reindex(columns=window_order, fill_value=0))
            cat_window_pct = cat_window.div(cat_window.sum(axis=1), axis=0) * 100
            cat_window_pct.plot(kind="bar", stacked=True, ax=ax, edgecolor="white")
            ax.set_title("Cascade speed regime distribution by device category (%)")
            ax.set_xlabel("Device category")
            ax.set_ylabel("% of cascade days")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
            ax.legend(title="Time window", bbox_to_anchor=(1.05,1), loc="upper left")
            plt.tight_layout()
            save_and_log(fig, "/tmp/ps2_window_by_device_cat.png", run)

        # ── 4. Summary stats table ────────────────────────────────────────────
        agg_cols = {"chain_length": ["mean","median","max"],
                    "chain_span_min": ["mean","median"],
                    "cascade_velocity_min_per_fault": ["mean"]}
        agg_cols = {k: v for k, v in agg_cols.items() if k in df.columns}
        summary = df.groupby("window_bucket", observed=True).agg(
            count=("chain_length","count"), **{
                f"{col}_{stat}": (col, stat)
                for col, stats_list in agg_cols.items()
                for stat in stats_list
            }
        ).reindex(window_order)
        summary.to_csv("/tmp/ps2_window_summary.csv")
        mlflow.log_artifact("/tmp/ps2_window_summary.csv")

        print("\n── Cascade Time-Window Summary ──────────────────────────────────────")
        print(summary.to_string())

        for w in window_order:
            if w in summary.index:
                mlflow.log_metric(f"window_{w.replace(' ','_')}_count",
                                  int(summary.loc[w,"count"]))
                if "chain_length_mean" in summary.columns:
                    mlflow.log_metric(f"window_{w.replace(' ','_')}_avg_chain_len",
                                      round(float(summary.loc[w,"chain_length_mean"]),3))

        print("\nTime-window analysis logged to MLflow.")

In [ ]:
# ── CELL 11 : Subsystem φ (Phi) Correlation Matrix ───────────────────────────
# Phi = Matthews Correlation Coefficient for two binary variables.
# For each pair of subsystems: does subsystem A failing predict subsystem B failing
# in the same cascade chain?  Computed overall and per device category.

if "subsystem_chain" not in df.columns:
    print("subsystem_chain not available — phi correlation skipped")
else:
    with mlflow.start_run(run_name="subsystem_phi_correlation") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "phi_correlation_matrix"})

        def compute_phi_matrix(df_sub, label="all"):
            subsystems = get_all_subsystems(df_sub)
            if len(subsystems) < 2:
                print(f"  [{label}] fewer than 2 subsystems — skipped")
                return None, subsystems
            flags = chain_to_binary_flags(df_sub, subsystems).astype(bool)
            phi_mat = pd.DataFrame(np.zeros((len(subsystems), len(subsystems))),
                                   index=subsystems, columns=subsystems)
            for s1, s2 in combinations(subsystems, 2):
                p = phi_corr(flags[s1], flags[s2])
                phi_mat.loc[s1, s2] = p
                phi_mat.loc[s2, s1] = p
            np.fill_diagonal(phi_mat.values, 1.0)
            return phi_mat, subsystems

        def plot_phi_matrix(phi_mat, title, path):
            fig, ax = plt.subplots(figsize=(max(8, len(phi_mat)*0.9),
                                            max(6, len(phi_mat)*0.8)))
            sns.heatmap(phi_mat, annot=True, fmt=".2f",
                        cmap="RdYlGn", center=0, vmin=-1, vmax=1,
                        ax=ax, linewidths=0.4, square=True)
            ax.set_title(title, fontsize=11)
            plt.tight_layout()
            save_and_log(fig, path, run)
            return path

        # ── Overall phi matrix ────────────────────────────────────────────────
        phi_all, subs_all = compute_phi_matrix(df, label="overall")
        if phi_all is not None:
            plot_phi_matrix(phi_all,
                "PS2 — Subsystem Phi Correlation (all devices, all cascade days)",
                "/tmp/ps2_phi_overall.png")
            phi_all.to_csv("/tmp/ps2_phi_overall.csv")
            mlflow.log_artifact("/tmp/ps2_phi_overall.csv")

            # Log strongest positive pairs
            phi_upper = phi_all.where(np.triu(np.ones(phi_all.shape), k=1).astype(bool))
            top_pairs = (phi_upper.stack()
                         .reset_index()
                         .rename(columns={"level_0":"sub_a","level_1":"sub_b",0:"phi"})
                         .sort_values("phi", ascending=False))
            top_pairs.to_csv("/tmp/ps2_phi_pairs.csv", index=False)
            mlflow.log_artifact("/tmp/ps2_phi_pairs.csv")

            print("\nTop 5 positively correlated subsystem pairs (phi):")
            print(top_pairs.head(5).to_string(index=False))
            for _, r in top_pairs.head(5).iterrows():
                key = f"phi_{r['sub_a']}__{r['sub_b']}".replace(" ","_")
                mlflow.log_metric(key, round(float(r["phi"]), 4))

        # ── Per device category ───────────────────────────────────────────────
        if "mars_device_category" in df.columns:
            for cat in df["mars_device_category"].dropna().unique():
                df_cat = df[df["mars_device_category"] == cat]
                phi_cat, _ = compute_phi_matrix(df_cat, label=cat)
                if phi_cat is not None:
                    path_cat = f"/tmp/ps2_phi_{cat.lower()}.png"
                    csv_cat  = f"/tmp/ps2_phi_{cat.lower()}.csv"
                    plot_phi_matrix(phi_cat,
                        f"PS2 — Subsystem Phi Correlation [{cat}]", path_cat)
                    phi_cat.to_csv(csv_cat)
                    mlflow.log_artifact(csv_cat)

        print("\nPhi correlation analysis logged to MLflow.")

In [ ]:
# ── CELL 12 : Conditional Probability P(B | A, T) ────────────────────────────
# Given that subsystem A fails, what is the probability that subsystem B also
# fails in the SAME cascade chain, broken down by time-window bucket T?
# P(B|A,T) = (#chains in window T that contain both A and B) /
#             (#chains in window T that contain A)
# Output: heatmap grid (one subplot per time window) + CSV + MLflow artifact.

if "subsystem_chain" not in df.columns or "window_bucket" not in df.columns:
    print("subsystem_chain or window_bucket not available — conditional prob skipped")
else:
    with mlflow.start_run(run_name="conditional_probability_PBgivenAT") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "conditional_probability_PBgivenAT"})

        subsystems = get_all_subsystems(df)
        window_order = list(TIME_WINDOWS.keys())
        n_sub = len(subsystems)

        # Build per-window P(B|A) matrices
        pb_given_a = {}
        for w in window_order:
            df_w = df[df["window_bucket"] == w]
            if len(df_w) == 0:
                pb_given_a[w] = pd.DataFrame(np.zeros((n_sub, n_sub)),
                                              index=subsystems, columns=subsystems)
                continue
            flags_w = chain_to_binary_flags(df_w, subsystems).astype(bool)
            mat = pd.DataFrame(np.zeros((n_sub, n_sub)),
                               index=subsystems, columns=subsystems)
            for i, sub_a in enumerate(subsystems):
                mask_a = flags_w[sub_a]
                n_a    = mask_a.sum()
                if n_a == 0:
                    continue
                for j, sub_b in enumerate(subsystems):
                    if i == j:
                        mat.iloc[i, j] = 1.0
                        continue
                    mat.iloc[i, j] = (mask_a & flags_w[sub_b]).sum() / n_a
            pb_given_a[w] = mat

        # Save all window matrices to one CSV
        all_rows = []
        for w, mat in pb_given_a.items():
            tmp = mat.stack().reset_index()
            tmp.columns = ["sub_a","sub_b","P_B_given_A"]
            tmp["window"] = w
            all_rows.append(tmp)
        cond_df = pd.concat(all_rows, ignore_index=True)
        cond_df.to_csv("/tmp/ps2_cond_prob.csv", index=False)
        mlflow.log_artifact("/tmp/ps2_cond_prob.csv")

        # Plot: one heatmap per time window
        n_windows = len(window_order)
        fig, axes = plt.subplots(1, n_windows,
                                  figsize=(min(5, 14//n_windows) * n_windows + 2,
                                           max(4, n_sub * 0.7)),
                                  sharey=True)
        if n_windows == 1:
            axes = [axes]

        for ax, w in zip(axes, window_order):
            mat = pb_given_a[w]
            sns.heatmap(mat, annot=(n_sub <= 10), fmt=".2f",
                        cmap="Blues", vmin=0, vmax=1,
                        ax=ax, cbar=(ax == axes[-1]),
                        xticklabels=True, yticklabels=(ax == axes[0]),
                        linewidths=0.3)
            ax.set_title(w, fontsize=9)
            ax.set_xlabel("")
            ax.tick_params(axis="x", rotation=90, labelsize=7)
            ax.tick_params(axis="y", labelsize=7)

        fig.suptitle("P(B | A, T) — Conditional probability subsystem B fails given A fails\n"
                     "(rows=A, cols=B, subplots=time window)", fontsize=11)
        plt.tight_layout()
        save_and_log(fig, "/tmp/ps2_cond_prob_grid.png", run)

        # Log top P(B|A) pairs across all windows
        top_cond = (cond_df[cond_df["sub_a"] != cond_df["sub_b"]]
                    .sort_values("P_B_given_A", ascending=False)
                    .head(10))
        print("\nTop 10 conditional probability pairs P(B|A,T):")
        print(top_cond.to_string(index=False))
        for _, r in top_cond.iterrows():
            key = (f"P_{r['sub_b']}_given_{r['sub_a']}_"
                   f"{r['window'].replace(' ','_').replace('+','p').replace('-','to')}")
            mlflow.log_metric(key, round(float(r["P_B_given_A"]), 4))

        print("\nConditional probability analysis logged to MLflow.")

In [ ]:
# ── CELL 13 : Markov Chain — cascade transition probabilities ─────────────────
# Builds a subsystem-to-subsystem transition probability matrix overall AND
# per device category (TVM / GATE / VALIDATOR).
# Gold grain = cascade days only — all rows in train_df are cascades.
# subsystem_chain separator = '->'

def build_markov(df_sub, chain_col="subsystem_chain"):
    """
    Returns (trans_prob np.ndarray, states list, n_chains int).
    trans_prob[i,j] = P(j | i) = probability state j follows state i.
    """
    chains = (df_sub[chain_col]
              .dropna()
              .apply(lambda s: [x.strip() for x in str(s).split("->") if x.strip()])
              .tolist())
    chains = [c for c in chains if len(c) >= 2]
    if not chains:
        return None, [], 0

    states    = sorted(set(s for c in chains for s in c))
    idx       = {s: i for i, s in enumerate(states)}
    n         = len(states)
    trans     = np.zeros((n, n))
    for chain in chains:
        for a, b in zip(chain[:-1], chain[1:]):
            if a in idx and b in idx:
                trans[idx[a], idx[b]] += 1
    row_sums = trans.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return trans / row_sums, states, len(chains)

if "subsystem_chain" not in df.columns:
    print("subsystem_chain not in Gold — Markov analysis skipped")
    MARKOV_RUN = None
else:
    with mlflow.start_run(run_name="markov_chain_analysis") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "markov_chain", "stage": "analysis"})

        # ── Overall transition matrix ─────────────────────────────────────────
        tp_all, states_all, n_chains_all = build_markov(train_df)
        if tp_all is not None:
            trans_df_all = pd.DataFrame(tp_all, index=states_all, columns=states_all)
            trans_df_all.to_csv("/tmp/markov_transition_all.csv")
            mlflow.log_artifact("/tmp/markov_transition_all.csv")
            mlflow.log_param("n_states_overall", len(states_all))
            mlflow.log_param("n_chains_overall", n_chains_all)

            fig, ax = plt.subplots(figsize=(max(8, len(states_all)*0.9),
                                             max(6, len(states_all)*0.8)))
            sns.heatmap(trans_df_all, annot=True, fmt=".2f",
                        cmap="YlOrRd", ax=ax, linewidths=0.5)
            ax.set_title("PS2 Markov Chain — Subsystem Cascade Transition P\n"
                         "(all devices, train split)")
            plt.tight_layout()
            save_and_log(fig, "/tmp/markov_transition_all.png", run)

        # ── Per device category ───────────────────────────────────────────────
        cat_results = {}
        if "mars_device_category" in df.columns:
            cats = train_df["mars_device_category"].dropna().unique()
            n_cats = len(cats)
            if n_cats > 0:
                fig_cat, axes_cat = plt.subplots(
                    1, n_cats,
                    figsize=(max(7, len(states_all)*0.8) * n_cats, max(5, len(states_all)*0.7)),
                    squeeze=False)
                for col_idx, cat in enumerate(sorted(cats)):
                    df_cat = train_df[train_df["mars_device_category"] == cat]
                    tp_cat, states_cat, n_chains_cat = build_markov(df_cat)
                    if tp_cat is None:
                        axes_cat[0, col_idx].set_title(f"[{cat}] — no data")
                        continue
                    trans_df_cat = pd.DataFrame(tp_cat, index=states_cat, columns=states_cat)
                    cat_results[cat] = trans_df_cat
                    trans_df_cat.to_csv(f"/tmp/markov_transition_{cat.lower()}.csv")
                    mlflow.log_artifact(f"/tmp/markov_transition_{cat.lower()}.csv")
                    mlflow.log_param(f"n_chains_{cat}", n_chains_cat)

                    sns.heatmap(trans_df_cat, annot=True, fmt=".2f",
                                cmap="YlOrRd", ax=axes_cat[0, col_idx],
                                linewidths=0.4, cbar=False,
                                xticklabels=True, yticklabels=True)
                    axes_cat[0, col_idx].set_title(f"[{cat}]", fontsize=10)
                    axes_cat[0, col_idx].tick_params(axis="x", rotation=90, labelsize=7)
                    axes_cat[0, col_idx].tick_params(axis="y", labelsize=7)

                fig_cat.suptitle("PS2 Markov Chain — Transition Probabilities by Device Category",
                                 fontsize=11)
                plt.tight_layout()
                save_and_log(fig_cat, "/tmp/markov_transition_by_cat.png", run)

        # ── Top cascade paths ─────────────────────────────────────────────────
        all_chains = (train_df["subsystem_chain"]
                      .dropna()
                      .apply(lambda s: [x.strip() for x in str(s).split("->") if x.strip()])
                      .tolist())
        path_counts = Counter(["->".join(c) for c in all_chains if len(c) >= 2])
        top_paths = pd.DataFrame(path_counts.most_common(10),
                                 columns=["path", "count"])
        top_paths.to_csv("/tmp/top_cascade_paths.csv", index=False)
        mlflow.log_artifact("/tmp/top_cascade_paths.csv")

        # Log significant overall transitions
        if tp_all is not None:
            for i, s1 in enumerate(states_all):
                for j, s2 in enumerate(states_all):
                    if tp_all[i, j] > 0.05:
                        mlflow.log_metric(f"trans_{s1}_to_{s2}", round(float(tp_all[i, j]), 3))

        MARKOV_RUN = run.info.run_id

    print(f"\nTop 5 cascade paths:")
    print(top_paths.head(5).to_string(index=False))
    if tp_all is not None:
        high_trans = [(states_all[i], states_all[j], tp_all[i, j])
                      for i in range(len(states_all)) for j in range(len(states_all))
                      if tp_all[i, j] > 0.15]
        high_trans.sort(key=lambda x: -x[2])
        print(f"\nHighest transition probabilities (overall):")
        for s1, s2, p in high_trans[:5]:
            print(f"  {s1} → {s2} : {p:.2%}")
    print(f"\nMarkov analysis logged (run={MARKOV_RUN})")

In [ ]:
# ── CELL 14 : HMM — 3-state Gaussian HMM on per-device cascade severity ───────
# Features: gold cascade-severity metrics (chain_length, critical_in_chain,
#            oos_in_chain, chain_span_min, distinct_subsystems).
# Grain is (device_id, transit_day) cascade days — sort by transit_day.
# States model severity progression: Minor → Moderate → Critical cascade.
# NOTE: all rows ARE cascade days (chain_length >= 2); HMM learns which cascade
# days cluster into low/medium/high severity regimes, not pre-vs-cascade.

HMM_FEATURES = [c for c in ["chain_length","critical_in_chain","oos_in_chain",
                              "chain_span_min","distinct_subsystems"]
                 if c in df.columns]

if len(HMM_FEATURES) >= 2:
    hmm_data = (train_df.sort_values(["DEVICE_ID","transit_day"])
                         .groupby("DEVICE_ID")[HMM_FEATURES]
                         .apply(lambda x: x.values.tolist())
                         .reset_index()
                         .rename(columns={0:"sequence"}))

    sequences = [np.array(seq) for seq in hmm_data["sequence"] if len(seq)>=3]
    if not sequences:
        print("No device has >= 3 cascade days in train split — HMM skipped")
        HMM_RUN = None
    else:
        X_hmm   = np.vstack(sequences)
        lengths = [len(s) for s in sequences]

        model_hmm = hmm.GaussianHMM(
            n_components=3, covariance_type="diag",
            n_iter=100, random_state=42, verbose=False)
        model_hmm.fit(X_hmm, lengths)

        with mlflow.start_run(run_name="hmm_3state") as run:
            mlflow.set_tags({**COMMON_TAGS,"model_type":"gaussian_hmm","n_states":"3"})
            mlflow.log_params({
                "n_components"   : 3,
                "covariance_type": "diag",
                "n_iter"         : 100,
                "hmm_features"   : str(HMM_FEATURES),
                "n_sequences"    : len(sequences),
            })
            for s in range(3):
                for fi, fn in enumerate(HMM_FEATURES):
                    mlflow.log_metric(f"state{s}_mean_{fn}",
                                       round(float(model_hmm.means_[s][fi]),4))

            trans_mat_df = pd.DataFrame(
                model_hmm.transmat_,
                columns=[f"→S{i}" for i in range(3)],
                index=[f"S{i}" for i in range(3)])
            trans_mat_df.to_csv("/tmp/hmm_transmat.csv")
            mlflow.log_artifact("/tmp/hmm_transmat.csv")

            joblib.dump(model_hmm, "/tmp/ps2_hmm.joblib")
            mlflow.log_artifact("/tmp/ps2_hmm.joblib","hmm_model")
            HMM_RUN = run.info.run_id

        print(f"\nHMM state means (cascade severity regime):")
        STATE_LABELS = ["Minor cascade","Moderate cascade","Critical cascade"]
        for s in range(3):
            print(f"  State {s} ({STATE_LABELS[s]}): "
                  f"{dict(zip(HMM_FEATURES,model_hmm.means_[s].round(3)))}")
        print(f"\nHMM transition matrix:\n{trans_mat_df.to_string()}")
else:
    print("Insufficient HMM features in Gold — skipped")
    HMM_RUN = None

In [ ]:
# ── CELL 15 : Association Rules — co-failing subsystem pairs ──────────────────
# Gold grain = (device_id, transit_day) cascade days only.
# Each row's subsystem_chain IS the basket (itemset) for that cascade day.
# subsystem_chain separator = '->'
# Runs overall and per device category.

def run_association_rules(df_sub, label="overall", min_support=0.05, min_lift=1.5):
    """
    Fit Apriori + Association Rules on subsystem_chain baskets.
    Returns (rules_out DataFrame, n_baskets, n_rules) or (None, 0, 0).
    """
    if "subsystem_chain" not in df_sub.columns:
        return None, 0, 0
    basket = [
        list(set(s.strip() for s in row.split("->") if s.strip()))
        for row in df_sub["subsystem_chain"].dropna()
    ]
    basket = [b for b in basket if len(b) >= 2]
    if not basket:
        print(f"  [{label}] No multi-item baskets — skipped")
        return None, 0, 0

    te     = TransactionEncoder()
    te_arr = te.fit(basket).transform(basket)
    bdf    = pd.DataFrame(te_arr, columns=te.columns_)

    try:
        freq   = apriori(bdf, min_support=min_support, use_colnames=True)
        if freq.empty:
            print(f"  [{label}] No frequent itemsets at support={min_support}")
            return None, len(basket), 0
        rules  = association_rules(freq, metric="lift", min_threshold=min_lift)
        rules  = rules.sort_values("lift", ascending=False)
        out    = rules[["antecedents","consequents","support","confidence","lift"]].copy()
        out["antecedents"] = out["antecedents"].apply(lambda x: ", ".join(sorted(x)))
        out["consequents"] = out["consequents"].apply(lambda x: ", ".join(sorted(x)))
        return out, len(basket), len(rules)
    except Exception as e:
        print(f"  [{label}] Association rules error: {e}")
        return None, len(basket), 0

if "subsystem_chain" not in df.columns:
    print("subsystem_chain not available — association rules skipped")
    RULES_RUN = None
else:
    with mlflow.start_run(run_name="association_rules") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "association_rules"})
        mlflow.log_params({"min_support": 0.05, "min_lift": 1.5})

        # ── Overall rules ─────────────────────────────────────────────────────
        rules_all, n_baskets_all, n_rules_all = run_association_rules(
            train_df, label="overall")
        mlflow.log_param("n_baskets_overall", n_baskets_all)
        mlflow.log_param("n_rules_overall",   n_rules_all)

        if rules_all is not None:
            rules_all.to_csv("/tmp/association_rules_all.csv", index=False)
            mlflow.log_artifact("/tmp/association_rules_all.csv")
            for i, row in rules_all.head(5).iterrows():
                mlflow.log_metric(f"rule_overall_{i}_lift", round(float(row["lift"]), 3))
            print(f"\nTop 5 association rules (overall):")
            print(rules_all.head(5).to_string(index=False))

        # ── Per device category ───────────────────────────────────────────────
        if "mars_device_category" in df.columns:
            for cat in sorted(train_df["mars_device_category"].dropna().unique()):
                df_cat = train_df[train_df["mars_device_category"] == cat]
                rules_cat, n_b, n_r = run_association_rules(df_cat, label=cat)
                mlflow.log_param(f"n_baskets_{cat}", n_b)
                mlflow.log_param(f"n_rules_{cat}",   n_r)

                if rules_cat is not None:
                    csv_path = f"/tmp/association_rules_{cat.lower()}.csv"
                    rules_cat.to_csv(csv_path, index=False)
                    mlflow.log_artifact(csv_path)
                    for i, row in rules_cat.head(3).iterrows():
                        mlflow.log_metric(f"rule_{cat}_{i}_lift",
                                          round(float(row["lift"]), 3))
                    print(f"\nTop 3 rules [{cat}]:")
                    print(rules_cat.head(3).to_string(index=False))

        RULES_RUN = run.info.run_id
    print(f"\nAssociation rules logged (run={RULES_RUN})")

In [ ]:
# ── CELL 16 : Network Graph — cascade relationship visualization ───────────────
# Directed graph: nodes = subsystems, edges = directional cascade transitions.
# Edge weight = transition count (from Markov chain raw counts).
# Node size  = total occurrence count across all cascade chains.
# Also builds one graph per device category for comparison.

def build_and_plot_network(df_sub, title, path, run=None, min_edge_weight=2):
    """
    Build a directed NetworkX graph from subsystem_chain transition counts.
    Nodes sized by total occurrence; edges sized/colored by transition count.
    """
    if "subsystem_chain" not in df_sub.columns:
        print(f"  subsystem_chain missing — network skipped ({title})")
        return None

    chains = (df_sub["subsystem_chain"]
              .dropna()
              .apply(lambda s: [x.strip() for x in str(s).split("->") if x.strip()])
              .tolist())
    chains = [c for c in chains if len(c) >= 2]
    if not chains:
        print(f"  No chains — network skipped ({title})")
        return None

    # Edge counts: A→B
    edge_counts = defaultdict(int)
    node_counts = Counter()
    for chain in chains:
        for s in chain:
            node_counts[s] += 1
        for a, b in zip(chain[:-1], chain[1:]):
            edge_counts[(a, b)] += 1

    G = nx.DiGraph()
    for node, cnt in node_counts.items():
        G.add_node(node, count=cnt)
    for (a, b), w in edge_counts.items():
        if w >= min_edge_weight:
            G.add_edge(a, b, weight=w)

    if G.number_of_edges() == 0:
        print(f"  No edges above min_weight={min_edge_weight} — network skipped ({title})")
        return G

    # Layout
    try:
        pos = nx.spring_layout(G, seed=42, k=2.0)
    except Exception:
        pos = nx.circular_layout(G)

    node_sizes  = [max(300, node_counts[n] * 50) for n in G.nodes()]
    edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_w = max(edge_weights) if edge_weights else 1
    edge_widths = [1 + 4 * (w / max_w) for w in edge_weights]
    edge_colors = [plt.cm.Reds(0.3 + 0.7 * w / max_w) for w in edge_weights]

    fig, ax = plt.subplots(figsize=(12, 9))
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                           node_color="#1565C0", alpha=0.85)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color="white",
                            font_weight="bold")
    nx.draw_networkx_edges(G, pos, ax=ax, width=edge_widths,
                           edge_color=edge_colors, arrows=True,
                           arrowsize=15, connectionstyle="arc3,rad=0.1",
                           min_source_margin=20, min_target_margin=20)
    # Edge weight labels on significant edges (top 10 by weight)
    top_edges = sorted(edge_counts.items(), key=lambda x: -x[1])[:10]
    edge_labels = {(a, b): str(w) for (a, b), w in top_edges
                   if G.has_edge(a, b)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                 ax=ax, font_size=7, label_pos=0.3)

    ax.set_title(title, fontsize=11, pad=12)
    ax.axis("off")
    plt.tight_layout()

    if run is not None:
        save_and_log(fig, path, run)
    else:
        fig.savefig(path, dpi=100, bbox_inches="tight")
        plt.show(); plt.close(fig)

    return G

if "subsystem_chain" not in df.columns:
    print("subsystem_chain not available — network graph skipped")
    NETWORK_RUN = None
else:
    with mlflow.start_run(run_name="cascade_network_graph") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "networkx_cascade_graph"})
        mlflow.log_param("min_edge_weight", 2)

        # ── Overall network ───────────────────────────────────────────────────
        G_all = build_and_plot_network(
            df, "PS2 — Cascade Subsystem Network (all devices, all cascade days)",
            "/tmp/ps2_network_all.png", run=run)

        if G_all is not None:
            mlflow.log_metric("network_n_nodes", G_all.number_of_nodes())
            mlflow.log_metric("network_n_edges", G_all.number_of_edges())
            if G_all.number_of_nodes() > 0:
                mlflow.log_metric("network_density", round(nx.density(G_all), 4))

            # Centrality metrics — which subsystems are cascade hubs?
            if G_all.number_of_nodes() > 1:
                in_cent  = nx.in_degree_centrality(G_all)
                out_cent = nx.out_degree_centrality(G_all)
                between  = nx.betweenness_centrality(G_all, weight="weight")
                centrality_df = pd.DataFrame({
                    "in_degree_centrality" : in_cent,
                    "out_degree_centrality": out_cent,
                    "betweenness_centrality": between,
                }).sort_values("betweenness_centrality", ascending=False)
                centrality_df.to_csv("/tmp/ps2_network_centrality.csv")
                mlflow.log_artifact("/tmp/ps2_network_centrality.csv")

                print("\nTop subsystem cascade hubs (betweenness centrality):")
                print(centrality_df.head(5).to_string())
                for node, row_c in centrality_df.head(5).iterrows():
                    mlflow.log_metric(f"centrality_btw_{node}",
                                      round(float(row_c["betweenness_centrality"]), 4))

        # ── Per device category ───────────────────────────────────────────────
        if "mars_device_category" in df.columns:
            for cat in sorted(df["mars_device_category"].dropna().unique()):
                df_cat = df[df["mars_device_category"] == cat]
                G_cat  = build_and_plot_network(
                    df_cat,
                    f"PS2 — Cascade Subsystem Network [{cat}]",
                    f"/tmp/ps2_network_{cat.lower()}.png",
                    run=run)
                if G_cat is not None:
                    mlflow.log_metric(f"network_{cat}_n_nodes", G_cat.number_of_nodes())
                    mlflow.log_metric(f"network_{cat}_n_edges", G_cat.number_of_edges())

        NETWORK_RUN = run.info.run_id
    print(f"\nNetwork graph analysis logged (run={NETWORK_RUN})")

In [ ]:
# ── CELL 17 : Summary ─────────────────────────────────────────────────────────
print("=" * 70)
print("PS2 — Cascade Chain Pattern Analysis — Complete")
print("=" * 70)
print(f"Gold table      : s3://{GOLD_BUCKET}/{GOLD_PREFIX}")
print(f"                  (cascade days only — chain_length >= 2)")
print(f"Analysis grain  : (device_id, transit_day) cascade days")
print(f"Architecture    : CUBIC-MARS-AWS-v6  |  Analysis-only (no prediction endpoint)")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Feature Store   : {FS_GROUP_NAME}")
print()
print("── Analysis modules completed ───────────────────────────────────────────")
print(f"  CELL 10  Cascade Propagation Time-Window Analysis")
print(f"           Buckets: {list(TIME_WINDOWS.keys())}")
print(f"           Outputs: chain length/velocity by window, device category breakdown")
print()
print(f"  CELL 11  Subsystem Phi (φ) Correlation Matrix")
print(f"           Phi = Matthews correlation for binary subsystem co-occurrence")
print(f"           Outputs: overall matrix + per device category heatmaps")
print()
print(f"  CELL 12  Conditional Probability P(B | A, T)")
print(f"           P(subsystem B fails | A fails, time-window T)")
print(f"           Outputs: 5-panel heatmap grid + CSV")
print()
print(f"  CELL 13  Markov Chain Transition Probabilities")
print(f"           P(next_subsystem | current_subsystem) overall + per category")
print(f"           MLflow run : {MARKOV_RUN}")
print()
print(f"  CELL 14  HMM — 3-state Gaussian cascade severity regimes")
print(f"           States: Minor / Moderate / Critical cascade")
print(f"           MLflow run : {HMM_RUN}")
print()
print(f"  CELL 15  Association Rules — co-failing subsystem pairs")
print(f"           Apriori min_support=0.05, min_lift=1.5, overall + per category")
print(f"           MLflow run : {RULES_RUN}")
print()
print(f"  CELL 16  Network Graph — cascade relationship visualization")
print(f"           Directed NetworkX graph with betweenness centrality")
print(f"           MLflow run : {NETWORK_RUN}")
print()
print("── MLflow artifacts logged ──────────────────────────────────────────────")
print("  ps2_eda.png                     — EDA 2x3 grid")
print("  ps2_window_chain_length.png     — time-window chain length")
print("  ps2_cascade_velocity.png        — velocity by window + category")
print("  ps2_window_by_device_cat.png    — window distribution by category")
print("  ps2_window_summary.csv          — time-window stats table")
print("  ps2_phi_overall.png/csv         — phi correlation matrix")
print("  ps2_phi_pairs.csv               — top subsystem pair correlations")
print("  ps2_cond_prob_grid.png/csv      — P(B|A,T) heatmap grid")
print("  markov_transition_all.png/csv   — overall Markov matrix")
print("  markov_transition_by_cat.png    — per category Markov matrices")
print("  top_cascade_paths.csv           — top 10 cascade paths by frequency")
print("  hmm_transmat.csv / ps2_hmm.joblib — HMM model artifacts")
print("  association_rules_all.csv       — overall association rules")
print("  ps2_network_all.png             — cascade network graph (all)")
print("  ps2_network_centrality.csv      — subsystem betweenness centrality")
print()
print("── Feasibility note ─────────────────────────────────────────────────────")
print("  15% gap: SVN_STAGE.CMDB_CI / CMDB_REL = 0 rows")
print("  Impact : shared_infra_device_count + dependency_depth features unavailable")
print("  Status : analysis proceeds on chain-derived features only (sufficient)")
print("=" * 70)